# Notebook 01 — Dataset Loading

## Purpose of this notebook
This notebook does the following:

1. Loads all **12 tables** supplied in the client extract.
2. Reviews each table against the business problem and states which tables are **relevant**, which are **used in part**, and which are **not used**, with reasons.
3. Loads only the relevant tables and describes the **instances**, the **classes** of the target and the **class imbalance**.
4. Audits the data-quality issues mentioned in the README markdown about (duplicates, inconsistent category spelling, missing values, mixed date formats) so that the preprocessing in Notebook 02 is driven by evidence.

In [ ]:
#--- Imports, paths and shared helpers ---
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

DATA_DIR = Path("data")          # client extract (12 CSV tables + README)
ARTIFACTS = Path("artifacts")    # cleaned data, split assignments, fitted preprocessor, models
RESULTS = Path("results")        # metrics, tables and figures
FIGS = RESULTS / "figures"
for _p in (ARTIFACTS, RESULTS, FIGS):
    _p.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TARGET = "activated"
ID = "account_id"


def read_table(name):
    """Read a client table exactly as supplied.

    keep_default_na=False stops pandas turning the literal category "None"
    (e.g. investor_experience_level = None, 1,250 rows) into a missing value.
    Only genuinely empty cells become NaN.
    """
    return pd.read_csv(DATA_DIR / f"{name}.csv", keep_default_na=False, na_values=[""])

print("Setup complete. Data folder:", DATA_DIR.resolve())

## Step 1 — Inventory all 12 tables
Before deciding what to use, we record the size and integrity of every table: row count, number of distinct accounts, and exact duplicate rows. Table 7 and Table 8 are *long* (several rows per account), so their row count exceeds the account count. 
Table 10 has at most one row per account but covers only the accounts that answered a survey.